In [2]:
import numpy as np
from moabb.datasets import BNCI2015_001
from moabb.paradigms import MotorImagery, LeftRightImagery
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from mne.decoding import CSP
from moabb.evaluations import CrossSubjectEvaluation
from sklearn.pipeline import make_pipeline
from scipy import signal
from scipy.io import loadmat
import os
import mne
import glob

In [3]:
# Define a causal bandpass filter function using a Butterworth design.
def causal_bandpass_filter(data, lowcut=8, highcut=30, fs=250, order=50):
    nyq = 0.5 * fs
    # Normalize the cutoff frequencies (Matlab's fir1 expects normalized cutoff frequencies
    low = lowcut / nyq
    high = highcut / nyq
    # Design the FIR filter. Note: order+1 coefficients are returned to match Matlab's fir1 which returns n+1 taps.
    b = signal.firwin(order + 1, [low, high], window='hamming', pass_zero=False)
    # Apply the filter causally using lfilter (this introduces a constant delay).
    filtered_data = signal.lfilter(b, [1.0], data)
    return filtered_data

In [4]:
# Define data directory for GDF files
data_dir = '/home/vishwa/eeg_tl/Recreating papers/BCI2b/BCICIV_2b_gdf'

# Lists to hold data for all subjects
train_active_X = []         # List to hold numpy arrays with shape (n_trials, n_channels, n_times) per subject
train_active_y = []         # List to hold event labels per subject
train_active_metadata = []  # List to hold event metadata per subject

# Define subject IDs (B01 to B09)
subjects = [f'B{subj:02d}' for subj in range(1, 10)]

for subj in subjects:
    # Define training sessions for this subject (e.g., B0101T.gdf, B0102T.gdf, B0103T.gdf)
    session_ids = ['01T'] #, '02T', '03T']
    subj_epochs_list = []

    for sess in session_ids:
        filename = os.path.join(data_dir, f'{subj}{sess}.gdf')
        
        # Check if file exists to avoid errors
        if not os.path.exists(filename):
            print(f"File {filename} not found, skipping.")
            continue
        
        # Load the GDF file
        raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False)
        
        # Extract events from annotations, mapping '769' to 1 (left) and '770' to 2 (right)
        event_id_mapping = {'769': 1, '770': 2}
        events, event_dict = mne.events_from_annotations(raw, event_id=event_id_mapping, verbose=False)
        
        # Select only EEG channels (C3, Cz, C4)
        print(raw.ch_names)
        eeg_channels = [ch for ch in raw.ch_names if ch in ['EEG:C3', 'EEG:Cz', 'EEG:C4']]
        if len(eeg_channels) != 3:
            print(f"Warning: Expected 3 EEG channels for {subj}{sess}, found {len(eeg_channels)}: {eeg_channels}")
        raw_eeg = raw.pick_channels(eeg_channels, verbose=False)
        
        # Define epoching parameters (consistent with Dataset 2a)
        tmin = 0.5  # seconds after cue onset
        tmax = 3.5  # seconds after cue onset
        
        # Create epochs
        epochs = mne.Epochs(
            raw_eeg,
            events,
            event_id={'left': 1, 'right': 2},
            tmin=tmin,
            tmax=tmax,
            baseline=None,  # No baseline correction, matching your 2a code
            preload=True,
            verbose=False
        )
        
        subj_epochs_list.append(epochs)
    
    # Skip subject if no sessions were processed
    if not subj_epochs_list:
        print(f"No valid sessions found for subject {subj}, skipping.")
        continue
    
    # Concatenate epochs across sessions for this subject
    if len(subj_epochs_list) > 1:
        subj_epochs = mne.concatenate_epochs(subj_epochs_list, verbose=False)
    else:
        subj_epochs = subj_epochs_list[0]
    
    # Get the epoch data
    subj_data = subj_epochs.get_data()
    
    # Apply causal bandpass filter to each trial and channel (matching Dataset 2a)
    n_trials, n_channels, n_times = subj_data.shape
    fs = raw.info['sfreq']  # Sampling frequency (250 Hz for Dataset 2b)
    subj_filtered_data = np.empty_like(subj_data)
    for trial in range(n_trials):
        for ch in range(n_channels):
            subj_filtered_data[trial, ch, :] = causal_bandpass_filter(
                subj_data[trial, ch, :],
                lowcut=8,   # Lower bound of sensorimotor rhythm
                highcut=30, # Upper bound of sensorimotor rhythm
                fs=fs,
                order=50    # Filter order
            )
    
    # Append processed data, labels, and metadata
    train_active_X.append(subj_filtered_data)
    train_active_y.append(subj_epochs.events[:, 2])  # Labels in third column (1 or 2)
    train_active_metadata.append(subj_epochs.events)
    
    # Print shape to verify
    print(f"Subject {subj}: Epoch data shape {subj_filtered_data.shape}")

print(f"Loaded data for {len(train_active_X)} subjects.")


# Lists to hold data for all subjects
eval_active_X = []         # List to hold numpy arrays with shape (n_trials, n_channels, n_times) per subject
eval_active_y = []         # List to hold event labels per subject
eval_active_metadata = []  # List to hold event metadata per subject

# Define subject IDs (B01 to B09)
subjects = [f'B{subj:02d}' for subj in range(1, 10)]

for subj in subjects:
    # Define training sessions for this subject (e.g., B0101T.gdf, B0102T.gdf, B0103T.gdf)
    session_ids = ['02T'] #, '02T', '03T']
    subj_epochs_list = []

    for sess in session_ids:
        filename = os.path.join(data_dir, f'{subj}{sess}.gdf')
        
        # Check if file exists to avoid errors
        if not os.path.exists(filename):
            print(f"File {filename} not found, skipping.")
            continue
        
        # Load the GDF file
        raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False)
        
        # Extract events from annotations, mapping '769' to 1 (left) and '770' to 2 (right)
        event_id_mapping = {'769': 1, '770': 2}
        events, event_dict = mne.events_from_annotations(raw, event_id=event_id_mapping, verbose=False)
        
        # Select only EEG channels (C3, Cz, C4)
        print(raw.ch_names)
        eeg_channels = [ch for ch in raw.ch_names if ch in ['EEG:C3', 'EEG:Cz', 'EEG:C4']]
        if len(eeg_channels) != 3:
            print(f"Warning: Expected 3 EEG channels for {subj}{sess}, found {len(eeg_channels)}: {eeg_channels}")
        raw_eeg = raw.pick_channels(eeg_channels, verbose=False)
        
        # Define epoching parameters (consistent with Dataset 2a)
        tmin = 0.5  # seconds after cue onset
        tmax = 3.5  # seconds after cue onset
        
        # Create epochs
        epochs = mne.Epochs(
            raw_eeg,
            events,
            event_id={'left': 1, 'right': 2},
            tmin=tmin,
            tmax=tmax,
            baseline=None,  # No baseline correction, matching your 2a code
            preload=True,
            verbose=False
        )
        
        subj_epochs_list.append(epochs)
    
    # Skip subject if no sessions were processed
    if not subj_epochs_list:
        print(f"No valid sessions found for subject {subj}, skipping.")
        continue
    
    # Concatenate epochs across sessions for this subject
    if len(subj_epochs_list) > 1:
        subj_epochs = mne.concatenate_epochs(subj_epochs_list, verbose=False)
    else:
        subj_epochs = subj_epochs_list[0]
    
    # Get the epoch data
    subj_data = subj_epochs.get_data()
    
    # Apply causal bandpass filter to each trial and channel (matching Dataset 2a)
    n_trials, n_channels, n_times = subj_data.shape
    fs = raw.info['sfreq']  # Sampling frequency (250 Hz for Dataset 2b)
    subj_filtered_data = np.empty_like(subj_data)
    for trial in range(n_trials):
        for ch in range(n_channels):
            subj_filtered_data[trial, ch, :] = causal_bandpass_filter(
                subj_data[trial, ch, :],
                lowcut=8,   # Lower bound of sensorimotor rhythm
                highcut=30, # Upper bound of sensorimotor rhythm
                fs=fs,
                order=50    # Filter order
            )
    
    # Append processed data, labels, and metadata
    eval_active_X.append(subj_filtered_data)
    eval_active_y.append(subj_epochs.events[:, 2])  # Labels in third column (1 or 2)
    eval_active_metadata.append(subj_epochs.events)
    
    # Print shape to verify
    print(f"Subject {subj}: Epoch data shape {subj_filtered_data.shape}")

print(f"Loaded data for {len(train_active_X)} subjects.")

/tmp/ipykernel_230531/1988317205.py:26: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False)


['EEG:C3', 'EEG:Cz', 'EEG:C4', 'EOG:ch01', 'EOG:ch02', 'EOG:ch03']
Subject B01: Epoch data shape (120, 3, 751)


/tmp/ipykernel_230531/1988317205.py:26: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False)


['EEG:C3', 'EEG:Cz', 'EEG:C4', 'EOG:ch01', 'EOG:ch02', 'EOG:ch03']
Subject B02: Epoch data shape (120, 3, 751)


/tmp/ipykernel_230531/1988317205.py:26: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False)


...

['EEG:C3', 'EEG:Cz', 'EEG:C4', 'EOG:ch01', 'EOG:ch02', 'EOG:ch03']
Subject B07: Epoch data shape (120, 3, 751)


/tmp/ipykernel_230531/1988317205.py:118: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False)


['EEG:C3', 'EEG:Cz', 'EEG:C4', 'EOG:ch01', 'EOG:ch02', 'EOG:ch03']
Subject B08: Epoch data shape (120, 3, 751)


/tmp/ipykernel_230531/1988317205.py:118: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False)


['EEG:C3', 'EEG:Cz', 'EEG:C4', 'EOG:ch01', 'EOG:ch02', 'EOG:ch03']
Subject B09: Epoch data shape (120, 3, 751)
Loaded data for 9 subjects.


In [21]:
n_components = 8
csp = CSP(n_components=n_components, reg=None, log=True, norm_trace=False)
lda = LinearDiscriminantAnalysis()

In [22]:
pipeline = Pipeline([
            ('CSP', CSP(n_components=n_components, reg=None, log=True, norm_trace=False)),
            ('LDA', LinearDiscriminantAnalysis())
        ])

        # Fit and predict
for i in range(len(train_active_X)):
    pipeline.fit(train_active_X[i], train_active_y[i])
    accuracy = pipeline.score(eval_active_X[i], eval_active_y[i])
    print(accuracy)

Computing rank from data with rank=None
    Using tolerance 5.2e-07 (2.2e-16 eps * 3 dim * 7.8e+08  max singular value)
    Estimated rank (data): 3
    data: rank 3 computed from 3 data channels with 0 projectors
Reducing data rank from 3 -> 3
Estimating class=1 covariance using EMPIRICAL
Done.
Estimating class=2 covariance using EMPIRICAL
Done.
0.575
Computing rank from data with rank=None
    Using tolerance 4.2e-07 (2.2e-16 eps * 3 dim * 6.3e+08  max singular value)
    Estimated rank (data): 3
    data: rank 3 computed from 3 data channels with 0 projectors
Reducing data rank from 3 -> 3
Estimating class=1 covariance using EMPIRICAL
Done.
Estimating class=2 covariance using EMPIRICAL
Done.
0.5416666666666666
Computing rank from data with rank=None
    Using tolerance 8.5e-07 (2.2e-16 eps * 3 dim * 1.3e+09  max singular value)
    Estimated rank (data): 3
    data: rank 3 computed from 3 data channels with 0 projectors
Reducing data rank from 3 -> 3
Estimating class=1 covariance us